In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the path
path = "/content/drive/MyDrive/Tutorial Session"

# Change the current working directory
os.chdir(path)

# Verify you are in the right spot
print("Current Directory:", os.getcwd())
print("Files in folder:", os.listdir())

import json
with open('token_data.json','r') as f:
    token_dict = json.load(f)

HF_TOKEN = token_dict['HF_TOKEN']
OPENAI_TOKEN = token_dict['OPENAI_TOKEN']

# Replace the following with your own credentials
# HF_TOKEN =
# OPENAI_TOKEN =

Mounted at /content/drive
Current Directory: /content/drive/.shortcut-targets-by-id/15YxA3hPvK35rfyNAcfskPdPdJVhhRSjh/Tutorial Session
Files in folder: ['token_data.json', 'data', 'wandb', '__pycache__', 'KMGPT_API.py', 'AutoCriteria Demo.ipynb', 'Old Slides', 'qwen-example-lora', 'Biop Article.gdoc', 'Multi-Agent Demo Project.ipynb', 'ENAR Session 1.gslides', 'ENAR Session 2.gslides', 'Section 3: Autonomous Agents.ipynb', 'ENAR Session 3.gslides', 'Section 1: Intro to LLM.ipynb', 'Section 2: Biomedical Domain Adaptation.ipynb']


In [2]:
from openai import OpenAI

# Initialize the client
client = OpenAI(api_key=OPENAI_TOKEN)

# Module 1: Prompt Engineering

## Zero-Shot Prompting

**Definition**: Interacting with an LLM by providing a task and context without any prior examples (shots) of the desired input-output mapping. The model relies entirely on the broad "world knowledge" it acquired during its multi-billion parameter pre-training.

Best Usage Scenarios in Biomedical Research:

**Simple Extraction**: Pulling standardized variables (Age, Sex, Primary Diagnosis) from unstructured discharge summaries.

**De-identification**: Identifying and masking basic PHI like names or addresses.

**Baseline Benchmarking**: Establishing a "floor" performance before investing in Few-Shot or Fine-Tuning.

### Example 1: Patient Information Extraction

In [3]:
clinical_note = """
Patient is a 64-year-old male with a history of T2DM and hypertension.
Admitted on 2024-05-12 for community-acquired pneumonia.
Treated with Ceftriaxone 1g IV daily. Discharge planned for 2024-05-17.
"""

prompt = f"""
You are a clinical data abstractor. Extract the following information from the note provided below.

Fields to extract:
1. Patient_Age
2. Primary_Diagnosis
3. Medication_Name
4. Admission_Date

Clinical Note:
{clinical_note}
"""

In [4]:
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=0)

print(response.choices[0].message.content)

1. Patient_Age: 64
2. Primary_Diagnosis: Community-acquired pneumonia
3. Medication_Name: Ceftriaxone
4. Admission_Date: 2024-05-12


## One-Shot/Few-Shot Learning

**Definition**: Providing the LLM with one (One-Shot) or several (Few-Shot) pairs of example inputs and outputs before the actual task.

**Why use it in Biomedicine?**



* Handling Ambiguity: Teaching the model how to handle conflicting information
* Complex Formatting: Enforcing strict data standards like CDISC or specific nested JSON structures.
* Edge Cases: Showing how to label "None" or "Unknown" when information is missing, which zero-shot models often hallucinate.

**Note**: Accuracy typically plateaus after 3–5 examples. Adding too many can consume the "context window" and increase costs.

### Example 2: Patient Information Extraction with One-Shot Learning

In [5]:
# The "Shot" (Example 1)
example_note = "Female, 45, history of HTN. Adm 03/12/23 for UTI. Tx: Cipro."
example_output = """{
  "Patient_Age": 45,
  "Primary_Diagnosis": "Urinary Tract Infection",
  "Medication_Name": "Ciprofloxacin",
  "Admission_Date": "2023-03-12"
}"""

# The "Target" (The actual case to solve)
test_note = """
Patient is a 64-year-old male with a history of T2DM and hypertension.
Admitted on 2024-05-12 for community-acquired pneumonia.
Treated with Ceftriaxone 1g IV daily. Discharge planned for 2024-05-17.
"""

prompt = f"""
You are a clinical data abstractor. Extract information into JSON.
Follow the formatting in the example below.

### EXAMPLE 1
Note: {example_note}
Output: {example_output}

### YOUR TASK
Note: {test_note}
Output:
"""

In [6]:
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=0)

print(response.choices[0].message.content)

```json
{
  "Patient_Age": 64,
  "Primary_Diagnosis": "Community-Acquired Pneumonia",
  "Medication_Name": "Ceftriaxone",
  "Admission_Date": "2024-05-12"
}
```


### Example 3: Extract Clinical Trial Information (AutoCriteria Demo)

**Reference**: Datta, Surabhi, et al. "AutoCriteria: a generalizable clinical trial eligibility criteria extraction system powered by large language models." Journal of the American Medical Informatics Association 31.2 (2024): 375-385.

In [7]:
criteria_text = '''
Inclusion Criteria:

Histologically- or cytologically-confirmed diagnosis of adenocarcinoma or squamous cell carcinoma of the esophagus or Siewert type I adenocarcinoma of the EGJ. Metastatic disease or locally advanced, unresectable disease. Life expectancy of greater than 3 months. Measurable disease based on Response Evaluation Criteria In Solid Tumors (RECIST) 1.1. Performance status of 0 or 1 on the Eastern Cooperative Oncology Group (ECOG) Performance Scale. Documented radiographic or clinical disease progression on no more or less than one previous line of standard therapy. Can provide either a newly obtained or archival tumor tissue sample for intra-tumoral immune-related testing and for anti-programmed cell death (PD)-1. Participants of reproductive potential must be willing to use adequate contraception for the course of the study through 120 days after the last dose of pembrolizumab or through 180 days after the last dose of paclitaxel, docetaxel or irinotecan. Adequate organ function
'''

# Kojima, Takashi, et al. "Randomized phase III KEYNOTE-181 study of pembrolizumab versus chemotherapy in advanced esophageal cancer." Journal of Clinical Oncology 38.35 (2020): 4138-4148.

In [8]:
inclusion_prompt_template = f"""
Please do not extract anything outside of the given Criteria Text. The extracted phrase spans should directly be from the given Criteria Text.

[Inclusion Criteria Text]:

{criteria_text}

From the above inclusion criteria text, identify the age, gender, education,
disease or non-alcoholic fatty liver disease (NAFLD) activity score (NAS) requirement, condition for steatosis score,
condition for Lobular inflammation score, condition for Ballooning degeneration score,
biomarkers (including liver fat content on magnetic resonance imaging proton density fat fraction or MRI-PDFF and fibrosis stage), lab tests, vitals (e.g., BMI), contraception-related criteria,
all prior treatments or therapies, medications or drugs, procedures (including imaging scans), disease diagnosis criteria, all other diseases or comorbidities, and life expectancy along with any conditions.
Extract all specific medication and drug names that are used to fight disease as well as medications for other diseases.
Extract all treatments and therapies including chemotherapy, immunotherapy, targeted therapy, inhibitors, or antibodies against molecules, and interventional studies.
Extract all comorbidities including all disease names (both hypernyms and their hyponyms), any health conditions, mental issues, complication, syndromes, disorder, symptoms, abnormalities, allergy, hypersensitivities, contraindication, adverse events, or side effects.

Show everything in 5 fields with keys Entity, Attribute, Value, Condition, and Sentence.

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
[
  {{
    "Entity": string, // the entity type of an eligibility criteria
    "Attribute": string, // the text span or phrase associated with an eligibility criteria
    "Value": string, // the value associated with an eligibility criteria
    "Condition": string, // any condition/restriction/exception or other details associated with an eligibility criteria
    "Sentence": string // the corresponding sentence or phrase in the text where an eligibility criteria was found
  }}
]

"""

In [9]:
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": inclusion_prompt_template}], temperature=0)

print(response.choices[0].message.content)

```json
[
  {
    "Entity": "Disease Diagnosis Criteria",
    "Attribute": "diagnosis of adenocarcinoma or squamous cell carcinoma of the esophagus or Siewert type I adenocarcinoma of the EGJ",
    "Value": "adenocarcinoma, squamous cell carcinoma, Siewert type I adenocarcinoma",
    "Condition": "Histologically- or cytologically-confirmed",
    "Sentence": "Histologically- or cytologically-confirmed diagnosis of adenocarcinoma or squamous cell carcinoma of the esophagus or Siewert type I adenocarcinoma of the EGJ."
  },
  {
    "Entity": "Disease Diagnosis Criteria",
    "Attribute": "Metastatic disease or locally advanced, unresectable disease",
    "Value": "Metastatic disease, locally advanced, unresectable disease",
    "Condition": "",
    "Sentence": "Metastatic disease or locally advanced, unresectable disease."
  },
  {
    "Entity": "Life Expectancy",
    "Attribute": "Life expectancy",
    "Value": "greater than 3 months",
    "Condition": "",
    "Sentence": "Life expectanc

# Module 2: Retrieval Augmented Generation

## Install Dependencies

In [10]:
!pip install -U langchain langchain-community langchain-openai pypdf langchain-experimental sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Chunking Example

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/Keynote-181.pdf")
data = loader.load()

## Recursive

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = splitter.split_documents(data)

print(f"Total chunks: {len(docs)}")
print(docs[0].metadata)
print(docs[0].page_content)

Total chunks: 67
{'producer': 'Acrobat Distiller 10.0.0 (Windows); modified using iText 4.2.0 by 1T3XT', 'creator': 'Arbortext Advanced Print Publisher 9.1.510/W Unicode', 'creationdate': '2020-11-18T15:39:23+05:30', 'keywords': '', 'crossmarkdomains[1]': 'http://ascopubs.org/', 'moddate': '2025-10-06T12:45:56-07:00', 'crossmarkmajorversiondate': '2020-11-18', 'subject': 'J Clin Oncol 2020.38:4138-4148', 'crossmarkdomains': '', 'author': '', 'title': 'Randomized Phase III KEYNOTE-181 Study of Pembrolizumab Versus Chemotherapy in Advanced Esophageal Cancer', 'crossmarkdomainexclusive': 'false', 'doi': '10.1200/JCO.20.01888', 'source': 'data/Keynote-181.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}
rapid communications
Randomized Phase III KEYNOTE-181 Study of
Pembrolizumab Versus Chemotherapy in
Advanced Esophageal Cancer
Takashi Kojima, MD 1; Manish A. Shah, MD 2; Kei Muro, MD 3; Eric Francois 4; Antoine Adenis, MD, PhD 5; Chih-Hung Hsu, MD, PhD 6;
Toshihiko Doi, MD, PhD 7; To

## Token-Based Splitting

In [ ]:
from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = splitter.split_documents(data)

print(f"Total chunks: {len(docs)}")
print(docs[0].metadata)
print(docs[0].page_content)

Total chunks: 45
{'producer': 'Acrobat Distiller 10.0.0 (Windows); modified using iText 4.2.0 by 1T3XT', 'creator': 'Arbortext Advanced Print Publisher 9.1.510/W Unicode', 'creationdate': '2020-11-18T15:39:23+05:30', 'keywords': '', 'crossmarkdomains[1]': 'http://ascopubs.org/', 'moddate': '2025-10-06T12:45:56-07:00', 'crossmarkmajorversiondate': '2020-11-18', 'subject': 'J Clin Oncol 2020.38:4138-4148', 'crossmarkdomains': '', 'author': '', 'title': 'Randomized Phase III KEYNOTE-181 Study of Pembrolizumab Versus Chemotherapy in Advanced Esophageal Cancer', 'crossmarkdomainexclusive': 'false', 'doi': '10.1200/JCO.20.01888', 'source': 'data/Keynote-181.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}
rapid communications
Randomized Phase III KEYNOTE-181 Study of
Pembrolizumab Versus Chemotherapy in
Advanced Esophageal Cancer
Takashi Kojima, MD 1; Manish A. Shah, MD 2; Kei Muro, MD 3; Eric Francois 4; Antoine Adenis, MD, PhD 5; Chih-Hung Hsu, MD, PhD 6;
Toshihiko Doi, MD, PhD 7; To

## Semantic Chunker

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create semantic splitter
splitter = SemanticChunker(embeddings)

# Apply to your PDF documents
docs = splitter.split_documents(data)

print(f"Total chunks: {len(docs)}")
print(docs[0].metadata)
print(docs[0].page_content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks: 31
{'producer': 'Acrobat Distiller 10.0.0 (Windows); modified using iText 4.2.0 by 1T3XT', 'creator': 'Arbortext Advanced Print Publisher 9.1.510/W Unicode', 'creationdate': '2020-11-18T15:39:23+05:30', 'keywords': '', 'crossmarkdomains[1]': 'http://ascopubs.org/', 'moddate': '2025-10-06T12:45:56-07:00', 'crossmarkmajorversiondate': '2020-11-18', 'subject': 'J Clin Oncol 2020.38:4138-4148', 'crossmarkdomains': '', 'author': '', 'title': 'Randomized Phase III KEYNOTE-181 Study of Pembrolizumab Versus Chemotherapy in Advanced Esophageal Cancer', 'crossmarkdomainexclusive': 'false', 'doi': '10.1200/JCO.20.01888', 'source': 'data/Keynote-181.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}
rapid communications
Randomized Phase III KEYNOTE-181 Study of
Pembrolizumab Versus Chemotherapy in
Advanced Esophageal Cancer
Takashi Kojima, MD 1; Manish A. Shah, MD 2; Kei Muro, MD 3; Eric Francois 4; Antoine Adenis, MD, PhD 5; Chih-Hung Hsu, MD, PhD 6;
Toshihiko Doi, MD, PhD 7; To

# Embedding Model Example

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/Keynote-181.pdf")
data = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = splitter.split_documents(data)

## TF-IDF

In [ ]:
from langchain_community.retrievers import TFIDFRetriever

# 1. Extract the text content from your Document objects
text_list = [doc.page_content for doc in docs]

# 2. Initialize the retriever from the texts
# This calculates the TF-IDF weights for all your chunks
retriever = TFIDFRetriever.from_texts(text_list)

# 3. Test a query
query = "What is the reported Overall Survival (OS) for this study?"
related_docs = retriever.invoke(query)

for i in range(len(related_docs)):
    print(f"Top {i+1} match content: \n {related_docs[i].page_content}")
    print('-------------------------------------------------------')

Top 1 match content: 
 preparation of the manuscript was provided by a medical
writer employed by the sponsor.
End Points
The three primary end points were overall survival (OS) in
patients with PD-L1 CPS $ 10, in patients with squamous
cell carcinoma, and in all patients. Secondary end points
CONTEXT
Key Objective
What is the antitumor activity of pembrolizumab versus chemotherapy as second-line treatment in patients with advanced or
metastatic esophageal cancer?
Knowledge Generated
Pembrolizumab provided a clinically meaningful survival bene ﬁt versus chemotherapy for patients with metastatic
esophageal squamous cell carcinoma and programmed death ligand-1 (PD-L1) combined positive score (CPS) $ 10
tumors and also in patients with metastatic esophageal squamous cell carcinoma or PD-L1 CPS $ 10 tumors, in the
second-line, with reduced toxicity.
Relevance
Pembrolizumab provided a clinically meaningful overall survival bene ﬁt versus chemotherapy in a global population of
--------------

# Demo

## Step 1: Load Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/Keynote-181.pdf")
data = loader.load()

## Step 2: Chunking Documents

In [ ]:
# We use the fixed-length chunking as a sliding window with overlaps.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

## Step 3: Create Embedding Library

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(api_key=OPENAI_TOKEN))

## Step 4: Inference with Built Library

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o", temperature=0, api_key=OPENAI_TOKEN)

# Add return_source_documents=True here
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Example Query
query = "What is the main efficacy endpoint in this paper?"
response = qa_chain.invoke(query)

# Output the LLM's Answer
print("--- LLM ANSWER ---")
print(response["result"])

# Output the Top 1 Retrieved Chunk
print("\n--- TOP RETRIEVED CHUNK ---")
if response["source_documents"]:
    top_doc = response["source_documents"][0]
    print(f"Content: {top_doc.page_content}")
    print(f"Metadata: {top_doc.metadata}")
else:
    print("No documents were retrieved.")

--- LLM ANSWER ---
The main efficacy endpoint in this paper is overall survival (OS) in patients with PD-L1 CPS ≥ 10, in patients with squamous cell carcinoma, and in all patients.

--- TOP RETRIEVED CHUNK ---
Content: proved by the appropriate institutional review board or
ethics committee at each participating institution. All au-
thors attest that the trial was conducted in accordance with
the Protocol and all its amendments and Good Clinical
Practice standards. All authors had full access to the data
and were involved in the writing or reviewing and editing
drafts of the manuscript and vouch for the accuracy and
completeness of the data analyses. Assistance in the
preparation of the manuscript was provided by a medical
writer employed by the sponsor.
End Points
The three primary end points were overall survival (OS) in
patients with PD-L1 CPS $ 10, in patients with squamous
cell carcinoma, and in all patients. Secondary end points
CONTEXT
Key Objective
What is the antitumor activit

# Module 3: Supervised Fine-tuning/Human Feedback Reinforcment Learning

# Supervised Fine-turning

## Install Dependencies

In [ ]:
!pip install trl

## Step 1: Set up a training set

In [ ]:
from datasets import Dataset

# 1. Your raw list of data
data_list = [
    {
        "messages": [
            {"role": "system", "content": "You are an Epidemiologist for HIV study."},
            {"role": "user", "content": '''
            What is the number of HIV diagnoses aged 13 years and older in the United States in 2025?
            Do not search the internet and give me the answer by yourself. '''},
            {"role": "assistant", "content": "There were 14,791 cases in 2025 from current CDC report."}
        ]
    }
]

# 2. Convert to a Hugging Face Dataset object (Fixes the AttributeError)
formatted_dataset = Dataset.from_list(data_list)

## Step 2: Load Tokenizer and Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset
import torch

model_id = "Qwen/Qwen3-0.6B"

# 1. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token = HF_TOKEN
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Step 3: Set Up Training

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    r=8,              # Rank of the update matrices
    lora_alpha=16,    # Scaling factor
    target_modules=["q_proj", "v_proj"], # Target the attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

# Setup Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen-example-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    num_train_epochs=40,
    save_steps=10,
    logging_steps=1,
    bf16=True # Use bfloat16 if hardware supports it
)

# Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
)

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

## Step 4: Train and Save Model

In [ ]:
trainer.train()
model.save_pretrained("data/final_lora_adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,4.493442
2,3.347689
3,2.563097
4,2.047824
5,1.651584
6,1.273348
7,0.949153
8,0.634793
9,0.397970
10,0.278326


In [ ]:
from peft import PeftModel

# 1. Load the base model again (cleanly)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Load your trained LoRA adapter
model = PeftModel.from_pretrained(base_model, "data/final_lora_adapter")

# 3. Merge the adapter weights into the base weights
merged_model = model.merge_and_unload()

# 4. Save the combined model
merged_model.save_pretrained("data/qwen3-demo-final")
tokenizer.save_pretrained("data/qwen3-demo-final")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('data/qwen3-demo-final/tokenizer_config.json',
 'data/qwen3-demo-final/chat_template.jinja',
 'data/qwen3-demo-final/tokenizer.json')

## Reload Trained Model

In [ ]:
from transformers import pipeline

# Initialize the pipeline with your TUNED model
pipe = pipeline(
    "text-generation",
    model="data/qwen3-demo-final",  # Path to your merged model
    torch_dtype="auto",
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
system_prompt = 'You are an Epidemiologist for HIV study.'

query = '''
What is the number of HIV diagnoses aged 13 years and older in the United States in 2025?
Do not search the internet and give me the answer by yourself.
'''

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": query}
]

# Generate the response
response = pipe(messages, max_new_tokens=512, temperature=0.1)

# Print the result
print(response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>

</think>

There were 14,791 cases in 2025 from current CDC report.
